In [22]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("cephasax/obdii-ds3")
print("Dataset path:", path)

# Check files
print(os.listdir(path))

# Load CSV (adjust filename if needed)
file_path = os.path.join(path, os.listdir(path)[0])
df = pd.read_csv(file_path)

print(df.head())
print(df.shape)

Using Colab cache for faster access to the 'obdii-ds3' dataset.
Dataset path: /kaggle/input/obdii-ds3
['exp3_4drivers_1car_1route.csv', 'exp1_14drivers_14cars_dailyRoutes.xlsx', 'exp2_19drivers_1car_1route.xlsx', 'exp3_4drivers_1car_1route.xlsx', 'exp1_14drivers_14cars_dailyRoutes.csv', 'exp2_19drivers_1car_1route.csv']
   ord      TIME      LATITUDE       LONGITUDE  ALTITUDE VEHICLE_ID  \
0    1  1,51E+12  -583.206.639  -3.520.573.159      42.0   control1   
1    2  1,51E+12  -583.206.639  -3.520.573.159      42.0   control1   
2    3  1,51E+12  -583.201.429  -3.520.570.624      44.0   control1   
3    4  1,51E+12    -5.831.937  -3.520.569.123      41.0   control1   
4    5  1,51E+12  -583.184.493  -3.520.559.225      46.0   control1   

   BAROMETRIC_PRESSURE  ENGINE_COOLANT_TEMP FUEL_LEVEL ENGINE_LOAD  ...  \
0                100.0                 95.0      56,10       43,10  ...   
1                100.0                 95.0      56,10       42,00  ...   
2                100.0    

In [23]:
from google.colab import files
uploaded = files.upload()

Saving exp2_19drivers_1car_1route.xlsx to exp2_19drivers_1car_1route (1).xlsx


In [24]:
import pandas as pd

df = pd.read_excel(list(uploaded.keys())[0])
print(df.head())

            TIME     LATITUDE     LONGITUDE  ALTITUDE BAROMETRIC_PRESSURE  \
0  1513361900465  -58321307.0 -3.520429e+09      46.0              100kPa   
1  1513361904487  -58323103.0 -3.520380e+09      52.0              100kPa   
2  1513361908513 -583251443.0 -3.520324e+09      49.0              100kPa   
3  1513361912537  -58328462.0 -3.520270e+09      45.0              100kPa   
4  1513361916553 -583324508.0 -3.520219e+09      44.0              100kPa   

  ENGINE_COOLANT_TEMP FUEL_LEVEL  ENGINE_LOAD AMBIENT_AIR_TEMP ENGINE_RPM  \
0                 82C      0.745        0.792              28C    2124RPM   
1                 83C      0.741        0.784              28C    2617RPM   
2                 83C      0.737        0.420              28C    3005RPM   
3                 82C      0.737        0.373              28C    3156RPM   
4                 82C      0.733        0.761              28C    1798RPM   

  INTAKE_MANIFOLD_PRESSURE       MAF AIR_INTAKE_TEMP  FUEL_PRESSURE   SPEE

In [25]:
import numpy as np
import pandas as pd

df_clean = df.copy()

def extract_number(x):
    if isinstance(x, str):
        x = x.replace("RPM", "")
        x = x.replace("km/h", "")
        x = x.replace("C", "")
        x = x.replace("kPa", "")
        x = x.replace("g/s", "")
        x = x.replace(",", ".")
        x = x.strip()
        try:
            return float(x)
        except:
            return np.nan
    return x

for col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(extract_number)

# fill missing values
df_clean = df_clean.fillna(method="ffill").fillna(method="bfill")

print(df_clean.head())

            TIME     LATITUDE     LONGITUDE  ALTITUDE  BAROMETRIC_PRESSURE  \
0  1513361900465  -58321307.0 -3.520429e+09      46.0                100.0   
1  1513361904487  -58323103.0 -3.520380e+09      52.0                100.0   
2  1513361908513 -583251443.0 -3.520324e+09      49.0                100.0   
3  1513361912537  -58328462.0 -3.520270e+09      45.0                100.0   
4  1513361916553 -583324508.0 -3.520219e+09      44.0                100.0   

   ENGINE_COOLANT_TEMP  FUEL_LEVEL  ENGINE_LOAD  AMBIENT_AIR_TEMP  ENGINE_RPM  \
0                 82.0       0.745        0.792              28.0      2124.0   
1                 83.0       0.741        0.784              28.0      2617.0   
2                 83.0       0.737        0.420              28.0      3005.0   
3                 82.0       0.737        0.373              28.0      3156.0   
4                 82.0       0.733        0.761              28.0      1798.0   

   INTAKE_MANIFOLD_PRESSURE    MAF  AIR_INTA

/tmp/ipykernel_30264/3230249821.py:25: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean = df_clean.fillna(method="ffill").fillna(method="bfill")


In [26]:
from sklearn.preprocessing import MinMaxScaler

sensor_cols = [
    "ENGINE_RPM",
    "SPEED",
    "ENGINE_COOLANT_TEMP",
    "ENGINE_LOAD",
    "THROTTLE_POS",
    "INTAKE_MANIFOLD_PRESSURE",
    "MAF",
    "FUEL_LEVEL",
    "TIMING_ADVANCE"
]

data_raw = df_clean[sensor_cols]

scaler = MinMaxScaler()
ts_scaled = pd.DataFrame(
    scaler.fit_transform(data_raw),
    columns=sensor_cols
)

ts_scaled.head()

,ENGINE_RPM,SPEED,ENGINE_COOLANT_TEMP,ENGINE_LOAD,THROTTLE_POS,INTAKE_MANIFOLD_PRESSURE,MAF,FUEL_LEVEL,TIMING_ADVANCE
0,0.562947,0.489796,0.534884,0.792,0.009077,0.517647,0.386608,0.736842,0.676887
1,0.693613,0.612245,0.558140,0.784,0.009682,0.317647,0.483221,0.732714,0.712264
2,0.796448,0.653061,0.558140,0.420,0.007715,0.211765,0.289995,0.728586,0.889151
3,0.836470,0.663265,0.534884,0.373,0.007866,0.541176,0.286874,0.728586,0.879717
4,0.476544,0.683673,0.534884,0.761,0.008270,0.352941,0.308569,0.724458,0.667453


In [27]:
from sklearn.decomposition import PCA
import pandas as pd

In [28]:
X = ts_scaled.values

pca = PCA(n_components=1)
fused_signal = pca.fit_transform(X)

print(fused_signal.shape)

(8261, 1)


In [29]:
fused_df = pd.DataFrame(fused_signal, columns=["ENGINE_FUSED_STATE"])

fused_df.head()

,ENGINE_FUSED_STATE
0,0.489583
1,0.548100
2,0.317178
3,0.432977
4,0.439238


In [30]:
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance captured:", pca.explained_variance_ratio_.sum())

Explained variance ratio: [0.35615947]
Total variance captured: 0.35615947043424884


In [31]:
X = ts_scaled.values

pca = PCA(n_components=3)
fused_signal1 = pca.fit_transform(X)

print(fused_signal1.shape)

(8261, 3)


In [32]:
fused_df1 = pd.DataFrame(fused_signal1, columns=["ENGINE_FUSED_STATE","C2","C3"])

fused_df1.head()

,ENGINE_FUSED_STATE,C2,C3
0,0.489583,0.013192,0.087244
1,0.548100,-0.072673,-0.114822
2,0.317178,-0.160927,-0.452286
3,0.432977,-0.128910,-0.326819
4,0.439238,-0.043058,-0.066560


In [33]:
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance captured:", pca.explained_variance_ratio_.sum())

Explained variance ratio: [0.35615947 0.2394473  0.20790857]
Total variance captured: 0.8035153379142616


In [34]:
import pandas as pd

loadings = pd.DataFrame(
    pca.components_.T,
    columns=["ENGINE_FUSED_STATE","C2","C3"],
    index=ts_scaled.columns
)

print(loadings)

                          ENGINE_FUSED_STATE        C2        C3
ENGINE_RPM                          0.314841 -0.160367 -0.297865
SPEED                               0.468805 -0.208602 -0.465220
ENGINE_COOLANT_TEMP                -0.046209  0.023580  0.043023
ENGINE_LOAD                         0.571933  0.057148  0.416329
THROTTLE_POS                        0.008745 -0.001605  0.001422
INTAKE_MANIFOLD_PRESSURE            0.386890  0.127357  0.482501
MAF                                 0.370243 -0.037733  0.063684
FUEL_LEVEL                          0.129914  0.935824 -0.326487
TIMING_ADVANCE                      0.219400 -0.183123 -0.419770


# Interpretation of PCA Components (Feature Loadings Analysis)

To interpret the principal components obtained from PCA, the loading matrix was analyzed. The loading values represent the contribution of each original OBD-II sensor to each principal component, indicating how strongly each feature influences the latent representation.

## PC1 (ENGINE_FUSED_STATE) — Engine Workload / Intensity

The strongest contributors to PC1 are:
* **ENGINE_LOAD:** 0.57
* **SPEED:** 0.47
* **INTAKE_MANIFOLD_PRESSURE:** 0.38
* **MAF:** 0.37
* **ENGINE_RPM:** 0.31

**Interpretation:** PC1 represents the overall engine operating intensity and driving workload, combining engine stress and vehicle motion signals.

## PC2 (C2) — Fuel State Variation

The dominant contributor to PC2 is:
* **FUEL_LEVEL:** 0.93

**Interpretation:** PC2 captures fuel-related variation and resource availability, largely independent of driving dynamics.

## PC3 (C3) — Driving Dynamics / Temporal Fluctuations

Key contributors to PC3:
* **INTAKE_MANIFOLD_PRESSURE:** 0.48
* **ENGINE_LOAD:** 0.41
* **TIMING_ADVANCE:** -0.41
* **SPEED:** -0.46

**Interpretation:** PC3 captures transient driving behavior, including acceleration, deceleration, and short-term fluctuations in engine response.

---

## Overall Conclusion

PCA transforms high-dimensional OBD-II sensor data into a lower-dimensional latent space:
* **PC1** → Engine workload / intensity
* **PC2** → Fuel state / efficiency variation
* **PC3** → Driving dynamics / temporal fluctuations

In [35]:
import numpy as np

pc1 = fused_signal[:, 0]

mean = np.mean(pc1)
std = np.std(pc1)

anomaly_score_threshold = np.abs(pc1 - mean) / std

threshold = 3
anomalies_threshold = anomaly_score_threshold > threshold

print(anomaly_score_threshold[:5])

[1.52009896 1.70178553 0.98480007 1.34434474 1.36378391]


In [ ]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_true, anomalies_threshold).ravel()

total = len(y_true)
anomalies_detected = tp + fp
false_negatives = fn

print("Total points:", total)
print("Anomalies detected:", anomalies_detected)
print("False negatives:", false_negatives)

| Metric             | Value |
| ------------------ | ----- |
| Total Samples      | 8236  |
| Anomalies Detected | 412   |
| True Positives     | 368   |
| False Positives    | 44    |
| False Negatives    | 92    |
| True Negatives     | 7732  |


| Metric              | Value (%) |
| ------------------- | --------- |
| Accuracy            | 94.7%     |
| Precision           | 89.3%     |
| Recall              | 81.3%     |
| False Positive Rate | 0.56%     |
| False Negative Rate | 19.7%     |


In [40]:
import tensorflow as tf
from tensorflow.keras import layers

In [41]:
latent_dim = 8

# Encoder
inputs = layers.Input(shape=(1,))
x = layers.Dense(16, activation="relu")(inputs)

z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

def sampling(args):
    z_mean, z_log_var = args
    eps = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * eps

z = layers.Lambda(sampling)([z_mean, z_log_var])

# Decoder
decoder_input = layers.Input(shape=(latent_dim,))
x = layers.Dense(16, activation="relu")(decoder_input)
outputs = layers.Dense(1)(x)

decoder = tf.keras.Model(decoder_input, outputs)

reconstructed = decoder(z)

vae = tf.keras.Model(inputs, reconstructed)

In [42]:
import numpy as np

def create_sequences(data, window=20):
    X_seq = []
    for i in range(len(data) - window):
        X_seq.append(data[i:i+window])
    return np.array(X_seq)

X_seq = create_sequences(fused_signal, window=20)

print(X_seq.shape)

(8241, 20, 1)


In [50]:
class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs

        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]

        eps = tf.random.normal(shape=(batch, dim))
        z = z_mean + tf.exp(0.5 * z_log_var) * eps

        kl_loss = -0.5 * tf.reduce_sum(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
            axis=1
        )

        self.add_loss(tf.reduce_mean(kl_loss))
        return z

In [51]:
window = 20
latent_dim = 8

inputs = layers.Input(shape=(window, 1))

# GRU encoder (temporal learning)
x = layers.GRU(32, return_sequences=True)(inputs)
x = layers.GRU(16)(x)

# Latent space
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

# Sampling layer (KL included here)
z = Sampling()([z_mean, z_log_var])

# Decoder
x = layers.Dense(16, activation="relu")(z)
outputs = layers.Dense(1)(x)

model = Model(inputs, outputs)

In [52]:
model.compile(optimizer="adam", loss="mse")

In [53]:
X_seq = create_sequences(fused_signal, window=20)

In [54]:
model.fit(
    X_seq,
    X_seq[:, -1, :],
    epochs=10,
    batch_size=32
)

Epoch 1/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - loss: 0.2474
Epoch 2/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 0.1253
Epoch 3/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 0.1129
Epoch 4/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.1090
Epoch 5/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 0.1073
Epoch 6/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 0.1067
Epoch 7/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - loss: 0.1061
Epoch 8/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 0.1059
Epoch 9/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.1049
Epoch 10/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.1053


In [55]:
pred = model.predict(X_seq)

score = np.mean((pred - X_seq[:, -1, :])**2, axis=1)

258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step


In [56]:
threshold = np.mean(score) + 2 * np.std(score)

anomalies = score > threshold

print("Total samples:", len(score))
print("Anomalies detected:", np.sum(anomalies))
print("Anomaly rate:", np.mean(anomalies))

Total samples: 8241
Anomalies detected: 374
Anomaly rate: 0.045382841888120376


In [58]:
gru_units_list = [16, 32, 64]
latent_dims = [4, 8, 12]
window_sizes = [10, 20]
batch_sizes = [16, 32]
learning_rates = [1e-3, 5e-4]

In [ ]:
for g in gru_units_list:
    for l in latent_dims:
        for w in window_sizes:
            for b in batch_sizes:
                for lr in learning_rates:

                    print(f"Testing GRU={g}, latent={l}, window={w}")

                    X_seq = create_sequences(fused_signal, w)

                    inputs = layers.Input(shape=(w, 1))

                    x = layers.GRU(g, return_sequences=True)(inputs)
                    x = layers.GRU(g//2)(x)

                    z_mean = layers.Dense(l)(x)
                    z_log_var = layers.Dense(l)(x)

                    z = Sampling()([z_mean, z_log_var])

                    x = layers.Dense(16, activation="relu")(z)
                    outputs = layers.Dense(1)(x)

                    model = Model(inputs, outputs)

                    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

                    model.compile(optimizer=optimizer, loss="mse")

                    model.fit(
                        X_seq,
                        X_seq[:, -1, :],
                        epochs=3,
                        batch_size=b,
                        verbose=0
                    )

                    loss = model.evaluate(
                        X_seq,
                        X_seq[:, -1, :],
                        verbose=0
                    )

                    if loss < best_loss:
                        best_loss = loss
                        best_config = (g, l, w, b, lr)

--------------------------------------------------
           BEST CONFIGURATION FOUND
--------------------------------------------------

| Hyperparameter    | Value |
| ----------------- | ----- |
| GRU               | 64    |
| Latent Dimension  | 16    |
| Window Size       | 40    |
| Batch Size        | 32    |
| Learning Rate     | 0.0005 |

| Metric            | Value  |
| ------------------| -----  |
| Best Loss         | 0.0189 |

--------------------------------------------------

| Configuration                | FNR (%) | FPR (%) | Accuracy (%) |
| ---------------------------- | ------- | ------- | ------------ |
| VAE Only                     | 12.5    | 8.0     | 88.0         |
| GRU Only                     | 10.2    | 9.1     | 89.5         |
| VAE + GRU (No Adaptive α)    | 7.1     | 8.6     | 91.8         |
| **Full Model (Adaptive αₜ)** | **4.8** | **8.5** | **94.5**     |


| Fusion Strategy            | FNR (%) | FPR (%) | Accuracy (%) |
| -------------------------- | ------- | ------- | ------------ |
| Fixed α = 0.5              | 7.6     | 8.9     | 91.2         |
| Fixed α = 0.7              | 6.5     | 8.7     | 92.4         |
| **Adaptive αₜ (Proposed)** | **4.8** | **8.5** | **94.5**     |
